# Load Antelope Tables
This is a prototype for a workflow to build a database of arrivals to drive s3 event extraction.   If this works out I plan to build the working data set from scratch driving the processing by either processing the raw table directly or giving students a csv file to drive the processing and giving them a revised version of this notebook to create it.  The later fits cleaner into existing notebook.  In fact, I think I might be able to pretty much fetch the same data.  

First, we have to instantiate and instance of the DatascopeDatabase class.

In [2]:
%env MSPASS_HOME=/opt/conda/lib/python3.10/site-packages/mspasspy
from mspasspy.preprocessing.css30.datascope import DatascopeDatabase
anfdb = DatascopeDatabase("css30/usarray_2010_03")

env: MSPASS_HOME=/opt/conda/lib/python3.10/site-packages/mspasspy


This does all the dirty wrk to build a single DataFrame of what we need including the "orid==prefor" clause.

In [3]:
df = anfdb.CSS30Catalog2df()

In [4]:
print(len(df))
print(df.keys())

134400
Index(['evid', 'evname', 'prefor', 'auth', 'commid', 'lddate', 'lat', 'lon',
       'depth', 'time', 'orid', 'jdate', 'nass', 'ndef', 'ndp', 'grn', 'srn',
       'etype', 'review', 'depdp', 'dtype', 'mb', 'mbid', 'ms', 'msid', 'ml',
       'mlid', 'algorithm', 'auth_origin', 'commid_origin', 'lddate_origin',
       'arid', 'sta', 'phase', 'belief', 'delta', 'seaz', 'esaz', 'timeres',
       'timedef', 'azres', 'azdef', 'slores', 'slodef', 'emares', 'wgt',
       'vmodel', 'commid_assoc', 'lddate_assoc', 'sta_arrival', 'time_arrival',
       'jdate_arrival', 'stassid', 'chanid', 'chan', 'iphase', 'stype',
       'deltim', 'azimuth', 'delaz', 'slow', 'delslo', 'ema', 'rect', 'amp',
       'per', 'logat', 'clip', 'fm', 'snr', 'qual', 'auth_arrival',
       'commid_arrival', 'lddate_arrival'],
      dtype='object')


## station metadata
We will need a list of stations eventually to fetch proper metadata.   For now this just validates an algorithm.

In [5]:
stalist = df['sta'].unique()

In [6]:
# examine result for validity
print(stalist)

['832A' '636A' '732A' '632A' '340A' '436A' '533A' '532A' '434A' '337A'
 '531A' '433A' 'JCT' '530A' '334A' '431A' '333A' '430A' '332A' '234A'
 '429A' '331A' '233A' '330A' '232A' '231A' '329A' '133A' 'ABTX' 'Z35A'
 '229A' '131A' '228A' 'Z32A' '129A' 'Y34A' 'Z31A' '128A' 'Z30A' 'Z29A'
 'Y32A' 'X34A' 'Y31A' 'X31A' 'W34A' 'TUL1' 'W33A' '319A' 'W32A' 'Y27A'
 'V34A' 'W31A' 'Y26A' 'V33A' 'V32A' 'Y25A' 'V31A' 'W29A' 'W27A' 'U32A'
 'TUC' 'W25A' 'T32A' 'V26A' 'ANMO' 'T28A' 'S29A' '113A' 'V22A' 'T25A'
 'S26A' 'X16A' 'T24B' 'U22A' 'S25A' 'R27A' 'R26A' 'S24A' 'SDCO' 'WUAZ'
 'R24A' 'S22A' 'MVCO' 'U15A' 'N26A' 'O23A' 'ECSD' 'N23A' 'DUG' 'G26A'
 'J20A' 'C30A' 'B29A' 'A29A' 'HLID' 'J23A' 'J24A' 'I24A' 'J22A' 'I22A'
 'RSSD' 'J25A' 'H23A' 'I25A' 'K22A' 'H24A' 'L23A' 'I21A' 'G23A' 'G22A'
 'F23A' 'F22A' 'F24A' 'G25A' 'F21A' 'H21A' 'F25A' 'H25A' 'E22A' 'H26A'
 'K21A' 'F20A' 'LAO' 'E24A' '833A' '534A' '529A' '230A' 'Z33A' '130A'
 'Y33A' 'MNTX' 'Z28A' 'Y29A' 'Y28A' 'Z26A' 'MSTX' 'Z25A' 'AMTX' 'Z24A'
 '121A' 'X

This month doesn't seem to have a net-sta issue but to make this more general one should handle that.   This little block builds a dictionary to cross-reference Antelope naming defined by their netsta table and net:sta used by Earthscope.  Note a complication is the method we called to create the giant dataframe doesn't merge netsta.

There is a strange issue with the nan station name.   That may cause problems downstream.

IMPORTANT:   I hacked in the snetsta table currently in the css30 directory.   The master data repository has arrival data in month directories with the dbmaster data somewhere else.   I'll need to figure that out eventually.  

In [7]:
df_netdata=anfdb.get_table('snetsta')
# useful to show this
print(df_netdata.columns)

Index(['snet', 'fsta', 'sta', 'lddate'], dtype='object')


Not obvious and explained only by digging into the antelope schema documents is that fsta is the seed sta name while "sta" for antelope is the composite string used to handle stations with duplicate station names and different net codes.  This little block demonstrates how many of them we have in the snetsta table we loaded.

In [8]:
print(df_netdata)

     snet  fsta   sta        lddate
0      AK   ANM   ANM  1.604521e+09
1      AK  BGLC  BGLC  1.604521e+09
2      AK   BMR   BMR  1.604521e+09
3      AK  BPAW  BPAW  1.604521e+09
4      AK  BRSE  BRSE  1.604521e+09
...   ...   ...   ...           ...
2264   TA  TFRD  TFRD  1.582128e+09
2265   TA  TPFO  TPFO  1.582128e+09
2266   TA  Y22D  Y22D  1.582128e+09
2267   TA  Y22E  Y22E  1.582128e+09
2268   TA  M65A  M65A  1.632863e+09

[2269 rows x 4 columns]


In [9]:
for row in df_netdata.itertuples():
    if row.fsta!=row.sta:
        print(row.fsta,row.sta)

Now that we know the difference between fsta and sta we can build this cross reference. 

In [10]:
netsta_xref=dict()
for row in df_netdata.itertuples():
    netsta_xref[row.sta] = [row.snet,row.fsta]

In [11]:
# print a couple examples and verify the length
print("size of cross-reference table=",len(netsta_xref))

size of cross-reference table= 2163


By combining the contents of stalist and netsta_xref we can build a bombproof algorithm to fetch station metadata with now standard web servcies.   

## Build subsetted arrivals DataFrame 
We next need to build a subset of the catalog dataframe that includes only teleseismic events.   For this case because the anf data are clean we can use the column of data with the key "delta", which in css3.0 means the distance in degrees.   That is a relatively standard DataFrame operation done in the next frame.

In [12]:
# there are multiple ways to do the following operation.  This is the clearest to humans
# this query uses stock limits for teleseismic P wave processing for receiver function deconvolution 
# which the tutorial workflow will be doing
# note prefor clause is needed because the method that created df doesn't do that according to the docstring
dftele = df.query('delta>30.0 and delta<100.0 and iphase=="P" and orid==prefor')
print("Size of original catalog = ",len(df))
print("Size of P wave subset = ",len(dftele))

Size of original catalog =  134400
Size of P wave subset =  83554


That is a reasonably large data set for a tutorial exercise with a group.   It would seem a monthly will work fine.  I will probably change the month eventually to be sure the data set contains some useful data, but this will certainly do for prototyping.   

I think the next step is pulling out only columns from the DataFrame that are essential to save as metadata.   This will also require using the cross reference even if we don't need it for this month of data.  First, here are the keys printed in a more readable form:

In [13]:
for k in dftele.keys():
    print(k)

evid
evname
prefor
auth
commid
lddate
lat
lon
depth
time
orid
jdate
nass
ndef
ndp
grn
srn
etype
review
depdp
dtype
mb
mbid
ms
msid
ml
mlid
algorithm
auth_origin
commid_origin
lddate_origin
arid
sta
phase
belief
delta
seaz
esaz
timeres
timedef
azres
azdef
slores
slodef
emares
wgt
vmodel
commid_assoc
lddate_assoc
sta_arrival
time_arrival
jdate_arrival
stassid
chanid
chan
iphase
stype
deltim
azimuth
delaz
slow
delslo
ema
rect
amp
per
logat
clip
fm
snr
qual
auth_arrival
commid_arrival
lddate_arrival


In [14]:

copylist=['evid','orid','arid','phase','delta','seaz','esaz','timeres','iphase','fm']
rename_list = { 'lat' : 'source_lat',
                'lon' : 'source_lon',
                'depth' : 'source_depth',
                'time' : 'source_time',
                'time_arrival' : 'Ptime',
                'chan' : 'pick_channel',
              }
magkeys=['ms','ml','mb']
# sta and net are handled specially using the cross reference
# note iterrows is not the fastest way to do this but good enough for 
# this application - cleaner syntax for sure for these data
doclist=list()
for index, row in dftele.iterrows():
    doc=dict()
    for k in copylist:
        doc[k] = row[k]
    for k in rename_list.keys():
        newname = rename_list[k]
        val=row[k]
        doc[newname] = val
    for k in magkeys:
        val = row[k]
        # null magnitudes are given a large negative value in antelope 
        # this needs to be more generic if reused in different context
        if val>4.0:   # could be 0 but appropriate for teleseism dataset
            doc[k] = val
    # handle neT:AR code
    css_sta = row.sta
    if css_sta in netsta_xref:
        x = netsta_xref[css_sta]
        net = x[0]
        sta = x[1]
        doc['net'] = net
        doc['sta'] = sta
    else:
        print(f"Warning:  {css_sta} not found in cross reference dictionary created from snetsta")
        print("sta is left unaltered and net is set to TA")
        doc["net"] = "TA"
    doclist.append(doc)

In [15]:
# QC results
from mspasspy.util.seismic import print_metadata
print("Size of arrival document list=",len(doclist))
print("Example document with pretty printing")
print_metadata(doclist[0])

Size of arrival document list= 83554
Example document with pretty printing
{
  "evid": 298930.0,
  "orid": 472052.0,
  "arid": 24259456,
  "phase": "P",
  "delta": 70.912,
  "seaz": 158.61,
  "esaz": 335.84,
  "timeres": -1.635,
  "iphase": "P",
  "fm": "-",
  "source_lat": -38.308,
  "source_lon": -73.916,
  "source_depth": 35.0,
  "source_time": 1267401686.22,
  "Ptime": 1267402358.16958,
  "pick_channel": "BHZ",
  "net": "TA",
  "sta": "832A"
}


## Save to MongoDB
I think the right approach here is to save these data to special mongodb collection.   We can read that later to drive waveform extraction from s3.

In [16]:
from mspasspy.client import Client
mspass_client=Client()
db = mspass_client.get_database("ES2026")

In [17]:
out = db.arrival_css30.insert_many(doclist)

In [18]:
# verify 
n=db.arrival_css30.count_documents({})
print("size of new collection created=",n)

size of new collection created= 83554


# Get Station Metadata
We know from lots of previous experience that downloading station metadata with obspy is best done as needed.   The data volume is relatively lightweight, the service runs fast, and there is a simple path to load the result to MongoDB found in all previous Earthscope tutorials.   

This version differs a bit from previously only in the way we select what metadata to download.   In this case we need to download station data for all networks for which we now have picks that where operational in the period of these data.  (January 2010).  First, let's get all net codes.

In [19]:
netlist=db.arrival_css30.distinct('net')
print(netlist)

['AZ', 'CI', 'IU', 'TA', 'US']


Now fetch and save metadata for all of those net codes.  

I'm using a generous time window but limiting the search region to a generous box around the lower 48 stations of the US.  That choice is because the focus here is stations the ANF used to produce their catalog.   

In [21]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
client=Client("Earthscope")

# time range around 6 months + or - Jan 2010
ts=UTCDateTime('2009-07-01T00:00:00.0')
starttime=ts
te=UTCDateTime('2012-07-01T00:00:00.0')
latitude_range=[20.0,55.0]
longitude_range=[-130.0,-60]
for net in netlist:
    inv=client.get_stations(network=net,
                            starttime=ts,
                            endtime=te,
                            minlatitude=latitude_range[0],
                            maxlatitude=latitude_range[1],
                            minlongitude=longitude_range[0],
                            maxlongitude=longitude_range[1],
                            format='xml',
                            channel='BH?',
                            level='response',
                           )
    print("Saving metadata for network=",net)
    db.save_inventory(inv)

Saving metadata for network= AZ
Database.save_inventory processing summary:
Number of site records processed= 16
number of site records saved= 0
number of channel records processed= 96
number of channel records saved= 0


KeyboardInterrupt: 

Finally, get a summary of how many documents were saved in channel and site.

In [22]:
nsite=db.site.count_documents({})
nchan=db.channel.count_documents({})
print("Number of documents in channel collection=",nchan)
print("Number of documents in site collection=",nsite)

Number of documents in channel collection= 6587
Number of documents in site collection= 1424


# Save source data
Saving source data is a bit different because I don't wish to save all the source data in the Datascope database.  We could and it wouldn't matter a lot but this is cleaner. 

In [23]:
df_src = anfdb.get_table("origin")
df_src = anfdb.join(df_src,"netmag",join_keys=["orid"])
print(df_src)

          lat       lon  depth          time    orid    evid    jdate  nass  \
0    -38.3080  -73.9160   35.0  1.267402e+09  472052  298930  2010060    95   
1    -38.3080  -73.9160   35.0  1.267402e+09  472052  298930  2010060    95   
2     43.5083 -105.2751    0.0  1.267403e+09  472053  298931  2010060    17   
3     44.4920 -105.6189    0.0  1.267403e+09  472054  298932  2010060    31   
4    -34.7140  -72.5600   38.4  1.267403e+09  472055  298933  2010060   220   
...       ...       ...    ...           ...     ...     ...      ...   ...   
1419  44.3753 -105.4652    0.0  1.270071e+09  473225  299906  2010090    54   
1420  43.7855 -105.2779    0.0  1.270073e+09  473226  299907  2010090    39   
1421  43.6910 -105.3329    0.0  1.270075e+09  473227  299908  2010090    52   
1422  45.0734 -106.9263    0.0  1.270077e+09  473228  299909  2010090    70   
1423  43.7372 -105.3370    0.0  1.270078e+09  473229  299910  2010090    46   

      ndef  ndp  ...   magid  net evid_netmag magty

In [24]:
# use this list to select only orid values of data used 
oridlist=db.arrival_css30.distinct("orid")
print("Number of events with arrivals saved=",len(oridlist))

Number of events with arrivals saved= 342


This next block that saves data is not at all generic.   It intentionally discards attributes from the css table that are not needed for this tutorial.

In [25]:
df_src.keys()

Index(['lat', 'lon', 'depth', 'time', 'orid', 'evid', 'jdate', 'nass', 'ndef',
       'ndp', 'grn', 'srn', 'etype', 'review', 'depdp', 'dtype', 'mb', 'mbid',
       'ms', 'msid', 'ml', 'mlid', 'algorithm', 'auth', 'commid', 'lddate',
       'magid', 'net', 'evid_netmag', 'magtype', 'nsta', 'magnitude',
       'uncertainty', 'auth_netmag', 'commid_netmag', 'lddate_netmag'],
      dtype='object')

In [26]:
doclist=list()
for index,row in df_src.iterrows():
    if row.orid in oridlist:
        doc = { "lat" : row.lat,
                "lon" : row.lon, 
                "depth" : row.depth,
                "time" : row.time, 
                "orid" : row.orid,
                "evid" : row.evid,
                "nass" : row.nass,
                "ndef" : row.ndef,
                "auth" : row.auth,
                "magnitude" : row.magnitude,
                "magtype" : row.magtype,
              }
        if row.ml>0.0:
            doc["ml"] = row.ml
        if row.mb>0.0:
            doc["mb"] = row.mb
        if row.ms>0.0:
            doc["ms"] = row.ms
        doclist.append(doc)
insert_result=db.source.insert_many(doclist)                


In [27]:
from mspasspy.util.seismic import print_metadata
n=db.source.count_documents({})
print("Number of source documents saved=",n)
print("Typical content")
doc=db.source.find_one()
print_metadata(doc)

Number of source documents saved= 547
Typical content
{
  "_id": {
    "$oid": "6a3278d2a2373042c271aca2"
  },
  "lat": -38.308,
  "lon": -73.916,
  "depth": 35.0,
  "time": 1267401686.22,
  "orid": 472052,
  "evid": 298930,
  "nass": 95,
  "ndef": 70,
  "auth": "QED_weekly",
  "magnitude": 4.8,
  "magtype": "ms"
}
